# BODAQS IMU Attitude Explorer - Self-scoped

Explore persisted offline IMU attitude streams from one BODAQS library. This notebook is intentionally limited to time-series and frequency-distribution inspection; it does not recompute the attitude estimate.

Reprocess a BDQ with the preprocess-profile `imu_attitude.enabled` setting before using this notebook.

## 1. Configure library

Set `LIBRARIES_ROOT` and `LIBRARY_ID`, then run the cells top-to-bottom.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import ipywidgets as W
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'bodaqs_analysis').is_dir():
            return candidate
        analysis = candidate / 'analysis'
        if (analysis / 'bodaqs_analysis').is_dir():
            return analysis
    raise RuntimeError('Could not find the BODAQS analysis package root from the current working directory.')


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / 'OneDrive' / 'BODAQS-data'
LIBRARY_ID = 'archie'

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item['library_id']: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ', '.join(sorted(libraries)) or 'none found'
    raise ValueError(f'Library {LIBRARY_ID!r} was not found. Available libraries: {available}')

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]['root'])
pio.renderers.default = 'notebook_connected'
print(f'Analysis package root: {ANALYSIS_DIR}')
print(f'Library root: {LIBRARY_ROOT}')

## 2. Select processed sessions

Only physical sessions are shown. Select one or more sessions, then use **Load selected attitude streams** below. Sessions without a persisted attitude stream are reported but ignored.

In [ ]:
from bodaqs_analysis.widgets.session_selector import make_session_selector

sel = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(sel['ui'])

## 3. Load an attitude stream

The stream selector identifies the session and IMU sensor. The inspector uses the persisted QC report in session metadata, so its status remains visible after reload.

In [ ]:
from bodaqs_analysis.artifacts import load_session_artifacts
from bodaqs_analysis.attitude import ATTITUDE_STREAM_SCHEMA

ATTITUDE_STREAMS = {}
stream_dd = W.Dropdown(description='Attitude stream', layout=W.Layout(width='720px'))
load_button = W.Button(description='Load selected attitude streams', button_style='primary')
load_status = W.Output(layout=W.Layout(width='100%'))


def _is_attitude_stream(name, metadata):
    return (
        str(name).startswith('attitude_')
        or isinstance(metadata, dict) and metadata.get('schema') == ATTITUDE_STREAM_SCHEMA
    )


def load_selected_attitude_streams(_=None):
    global ATTITUDE_STREAMS
    loaded, unavailable = {}, []
    for ref in sel['get_selected']():
        artifacts = load_session_artifacts(sel['store'], run_id=ref['run_id'], session_id=ref['session_id'])
        stream_dfs = artifacts.get('stream_dfs', {})
        stream_meta = artifacts.get('secondary_stream_meta', {})
        found = False
        for name, frame in stream_dfs.items():
            metadata = stream_meta.get(name, {})
            if _is_attitude_stream(name, metadata):
                key = f"{ref['session_id']} — {name}"
                loaded[key] = {
                    'df': frame.copy(),
                    'metadata': metadata,
                    'session_meta': artifacts.get('meta', {}),
                    'run_id': ref['run_id'],
                    'session_id': ref['session_id'],
                    'stream_name': name,
                }
                found = True
        if not found:
            unavailable.append(ref['session_id'])

    ATTITUDE_STREAMS = loaded
    stream_dd.options = list(loaded)
    if loaded and '_show_stream_metadata' in globals():
        _show_stream_metadata()
    with load_status:
        load_status.clear_output()
        print(f'Loaded {len(loaded)} attitude stream(s).')
        if unavailable:
            print('No persisted attitude stream: ' + ', '.join(unavailable))
            print('Reprocess these sessions with imu_attitude.enabled = true.')


load_button.on_click(load_selected_attitude_streams)
display(W.VBox([W.HBox([load_button, stream_dd]), load_status]))

## 4. Inspect time series and distributions

`*_rad` attitude and innovation signals are displayed in degrees. The time-series renderer preserves continuity-segment breaks and limits only its displayed samples; distribution statistics use every finite sample in the selected time window.

In [ ]:
signal_select = W.SelectMultiple(description='Signals', rows=12, layout=W.Layout(width='430px'))
start_time = W.FloatText(description='Start [s]', layout=W.Layout(width='190px'))
end_time = W.FloatText(description='End [s]', layout=W.Layout(width='190px'))
max_samples = W.BoundedIntText(value=12000, min=500, max=200000, step=500, description='Plot samples')
histogram_bins = W.BoundedIntText(value=80, min=10, max=500, step=10, description='Histogram bins')
plot_button = W.Button(description='Render selected views', button_style='primary')
metadata_out = W.Output(layout=W.Layout(width='100%'))
time_series_out = W.Output(layout=W.Layout(width='100%'))
distribution_out = W.Output(layout=W.Layout(width='100%'))


def _numeric_signals(frame):
    excluded = {'time_s', 'continuity_segment'}
    return [name for name in frame.columns if name not in excluded and pd.api.types.is_numeric_dtype(frame[name])]


def _display_signal(name, values):
    values = pd.to_numeric(values, errors='coerce')
    if name.endswith('_rad'):
        return name[:-4] + ' [deg]', np.degrees(values)
    return name, values


def _window(frame):
    if frame.empty or 'time_s' not in frame:
        return frame.iloc[0:0].copy()
    time = pd.to_numeric(frame['time_s'], errors='coerce')
    return frame.loc[time.between(start_time.value, end_time.value, inclusive='both')].copy()


def _display_sample(frame, limit):
    if len(frame) <= limit:
        return frame
    if 'continuity_segment' not in frame:
        return frame.iloc[np.linspace(0, len(frame) - 1, limit, dtype=int)]
    groups = list(frame.groupby('continuity_segment', sort=False))
    minimum = max(2, limit // max(1, len(groups)))
    parts = []
    for _, group in groups:
        count = min(len(group), max(minimum, round(limit * len(group) / len(frame))))
        parts.append(group.iloc[np.linspace(0, len(group) - 1, count, dtype=int)])
    return pd.concat(parts, ignore_index=True)


def _show_stream_metadata(change=None):
    if not stream_dd.value or stream_dd.value not in ATTITUDE_STREAMS:
        return
    item = ATTITUDE_STREAMS[stream_dd.value]
    frame = item['df']
    time = pd.to_numeric(frame['time_s'], errors='coerce')
    signals = _numeric_signals(frame)
    defaults = [name for name in ('roll_rad', 'pitch_rad', 'yaw_enu_rad', 'yaw_sigma_deg', 'course_innovation_rad') if name in signals]
    signal_select.options = signals
    signal_select.value = tuple(defaults or signals[:min(4, len(signals))])
    start_time.value = float(time.min())
    end_time.value = float(time.max())
    attitude_qc = item['session_meta'].get('attitude_qc', {})
    sensor = item['metadata'].get('sensor', item['stream_name'].removeprefix('attitude_'))
    qc = attitude_qc.get(sensor, {}) if isinstance(attitude_qc, dict) else {}
    with metadata_out:
        metadata_out.clear_output()
        print(f"Run: {item['run_id']} | Session: {item['session_id']} | Stream: {item['stream_name']}")
        print(f"Samples: {len(frame):,} | Time: {time.min():.3f} to {time.max():.3f} s")
        print(f"Attitude status: {qc.get('status', 'not recorded')} | Yaw observed fraction: {qc.get('yaw_observed_fraction', float('nan')):.3f}")
        print(f"Gravity updates accepted: {qc.get('gravity_updates_accepted', 'not recorded')} | Course updates accepted: {qc.get('course_updates_accepted', 'not recorded')}")


def render_selected_views(_=None):
    if not stream_dd.value or stream_dd.value not in ATTITUDE_STREAMS:
        raise RuntimeError('Load and select an attitude stream first.')
    selected_signals = list(signal_select.value)
    if not selected_signals:
        raise ValueError('Select at least one signal.')
    frame = _window(ATTITUDE_STREAMS[stream_dd.value]['df'])
    if frame.empty:
        raise ValueError('The selected time range contains no attitude samples.')
    plotted = _display_sample(frame, max_samples.value)

    with time_series_out:
        time_series_out.clear_output()
        figure = make_subplots(rows=len(selected_signals), cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=selected_signals)
        group_column = 'continuity_segment' if 'continuity_segment' in plotted else None
        for row, signal in enumerate(selected_signals, start=1):
            label, values = _display_signal(signal, plotted[signal])
            groups = plotted.groupby(group_column, sort=False) if group_column else [(None, plotted)]
            for _, group in groups:
                _, group_values = _display_signal(signal, group[signal])
                figure.add_trace(go.Scattergl(x=group['time_s'], y=group_values, mode='lines', name=label, legendgroup=signal, showlegend=(row == 1)), row=row, col=1)
            figure.update_yaxes(title_text=label, row=row, col=1)
        figure.update_xaxes(title_text='Time [s]', row=len(selected_signals), col=1)
        figure.update_layout(height=max(360, 260 * len(selected_signals)), title=f'Time series — {stream_dd.value}', hovermode='x unified')
        figure.show()

    with distribution_out:
        distribution_out.clear_output()
        figure = make_subplots(rows=1, cols=2, subplot_titles=('Frequency distribution', 'Empirical cumulative distribution'))
        for signal in selected_signals:
            label, values = _display_signal(signal, frame[signal])
            values = np.asarray(values, dtype=float)
            values = values[np.isfinite(values)]
            if not len(values):
                continue
            figure.add_trace(go.Histogram(x=values, nbinsx=histogram_bins.value, histnorm='probability density', name=label, opacity=0.55), row=1, col=1)
            sorted_values = np.sort(values)
            figure.add_trace(go.Scattergl(x=sorted_values, y=np.arange(1, len(sorted_values) + 1) / len(sorted_values), mode='lines', name=label, legendgroup=signal, showlegend=False), row=1, col=2)
        figure.update_xaxes(title_text='Signal value', row=1, col=1)
        figure.update_xaxes(title_text='Signal value', row=1, col=2)
        figure.update_yaxes(title_text='Density', row=1, col=1)
        figure.update_yaxes(title_text='Cumulative fraction', range=[0, 1], row=1, col=2)
        figure.update_layout(height=480, barmode='overlay', title=f'Distributions — {start_time.value:.3f} to {end_time.value:.3f} s')
        figure.show()


stream_dd.observe(_show_stream_metadata, names='value')
plot_button.on_click(render_selected_views)
controls = W.VBox([
    W.HBox([signal_select, W.VBox([start_time, end_time, max_samples, histogram_bins, plot_button])]),
    metadata_out,
    time_series_out,
    distribution_out,
])
display(controls)

## Reading the first-slice attitude signals

- `roll_rad`, `pitch_rad`, and `yaw_enu_rad` are the body-to-world estimate. Yaw is world-frame only while the state is `world_enu_constrained` or `world_enu_degraded`.
- `yaw_sigma_deg` shows the propagated yaw uncertainty.
- `gravity_update_weight`, `course_update_weight`, innovations, and rejection codes show which corrections were usable.
- `attitude_state_code`: `0` gravity aligned; `1` world ENU constrained; `2` world ENU degraded after a prior course correction.

For definitions and acceptance rules, see `docs/analysis/BMI270_Attitude_Derived_Product.md`.